# Chapter 5 — Explainability (SHAP and LIME) for the Hybrid Model

This notebook applies post-hoc explainability methods to the **final hybrid model** artifacts produced in **Notebook 05 (Hybrid Method)**. The goal is to interpret the drivers of multi-horizon inflation forecasts (t+1, t+3, t+5) using two complementary approaches: **SHAP** (global + local feature attribution) and **LIME** (local surrogate explanations).

## SHAP implementation (key points)

- **What SHAP explains:** SHAP decomposes each prediction into additive feature contributions relative to a baseline, consistent with a cooperative game-theoretic interpretation.
- **Model compatibility:** For tree-based models (XGBoost), we use **TreeSHAP** via `shap.TreeExplainer`, which is efficient and well-suited to boosted trees.
- **Horizon-wise explanations:** Because each horizon (t+1, t+3, t+5) corresponds to a separate fitted model artifact, SHAP is computed **separately per horizon**.
- **Global importance:** We report mean absolute SHAP values to quantify overall feature influence for each horizon.
- **Local behaviour:** Beeswarm plots highlight how feature values relate to positive/negative contributions across the test instances.
- **Reproducibility:** SHAP plots and feature-importance tables are saved under `artifacts/explainability/shap/`.

## LIME implementation (key points)

- **What LIME explains:** LIME explains an individual prediction by fitting a **local linear surrogate** model around the instance using perturbed samples.
- **Horizon-wise explanations:** As with SHAP, LIME explanations are produced **per horizon** because the underlying model differs.
- **Model interface:** LIME requires a prediction function; we wrap the saved XGBoost model artifacts into a callable that returns numeric predictions.
- **Outputs:** For a small number of representative test instances, we save LIME explanations as **HTML reports** under `artifacts/explainability/lime/`.

## Artifact provenance requirement

This notebook **does not train** models. It first checks that all Notebook 05 artifacts exist and records the file timestamps to confirm the explainability results correspond to the intended (latest) saved models and inputs.

In [5]:
# Artifact check — Notebook 05 (Hybrid Method)
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "results").exists() and (p / "src").exists():
            return p
    return start

def _fmt_mtime(path: Path) -> str:
    ts = datetime.fromtimestamp(path.stat().st_mtime)
    return ts.strftime("%Y-%m-%d %H:%M:%S")

def artifact_status(paths: list[Path], label: str, upstream_notebook: Path | None = None) -> pd.DataFrame:
    nb_exists = bool(upstream_notebook and upstream_notebook.exists())
    nb_mtime = upstream_notebook.stat().st_mtime if nb_exists else None
    nb_mtime_str = _fmt_mtime(upstream_notebook) if nb_exists else None

    rows: list[dict] = []
    for p in paths:
        exists = p.exists()
        art_mtime = p.stat().st_mtime if exists else None
        rows.append(
            {
                "group": label,
                "path": str(p),
                "exists": bool(exists),
                "modified": _fmt_mtime(p) if exists else None,
                "size_kb": round(p.stat().st_size / 1024, 1) if exists else None,
                "upstream_notebook": str(upstream_notebook) if upstream_notebook else None,
                "upstream_notebook_modified": nb_mtime_str,
                "artifact_newer_than_upstream": (art_mtime >= nb_mtime) if (exists and nb_exists) else None,
            }
        )
    return pd.DataFrame(rows)

PROJECT_ROOT = find_project_root(Path.cwd())
print("Project root:", PROJECT_ROOT)

NOTEBOOK_05 = PROJECT_ROOT / "notebooks" / "05_Hypertune_hybrid_method.ipynb"
HYBRID_DIR = PROJECT_ROOT / "results" / "hybrid"

# Required artifacts for explainability (saved by Notebook 05 workflow)
required_artifacts = [
    HYBRID_DIR / "xgb_t1.json",
    HYBRID_DIR / "xgb_t3.json",
    HYBRID_DIR / "xgb_t5.json",
    HYBRID_DIR / "stack_test_X.npy",
    HYBRID_DIR / "xgb_feature_names.json",
    HYBRID_DIR / "xgb_model_config.json",
    HYBRID_DIR / "xgb_summary.json",
]

df05 = artifact_status(required_artifacts, label="Notebook 05 — Hybrid", upstream_notebook=NOTEBOOK_05)
display(df05)

missing = df05.loc[~df05["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError("Missing Notebook 05 artifacts required for SHAP/LIME:\n" + "\n".join(missing))

if NOTEBOOK_05.exists():
    stale = df05.loc[df05["artifact_newer_than_upstream"] == False, "path"].tolist()  # noqa: E712
    if stale:
        print("WARNING: Some required artifacts are older than Notebook 05. If unexpected, re-run Notebook 05 before explainability.")
        for p in stale:
            print(" -", p)

print("All required Notebook 05 artifacts for explainability are present.")

Project root: C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI


,group,path,exists,modified,size_kb,upstream_notebook,upstream_notebook_modified,artifact_newer_than_upstream
0,Notebook 05 — Hybrid,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,True,2025-12-31 01:11:01,488.7,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,2026-01-03 03:55:50,False
1,Notebook 05 — Hybrid,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,True,2025-12-31 01:11:01,592.0,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,2026-01-03 03:55:50,False
2,Notebook 05 — Hybrid,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,True,2025-12-31 01:11:01,705.7,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,2026-01-03 03:55:50,False
3,Notebook 05 — Hybrid,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,True,2025-12-31 01:11:01,2.8,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,2026-01-03 03:55:50,False
4,Notebook 05 — Hybrid,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,True,2026-01-03 01:04:11,0.3,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,2026-01-03 03:55:50,False
5,Notebook 05 — Hybrid,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,True,2026-01-03 01:04:11,0.9,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,2026-01-03 03:55:50,False
6,Notebook 05 — Hybrid,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,True,2026-01-03 01:04:11,0.2,C:\Users\reset\OneDrive - UWE Bristol\Master_p...,2026-01-03 03:55:50,False


 - C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\xgb_t1.json
 - C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\xgb_t3.json
 - C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\xgb_t5.json
 - C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\stack_test_X.npy
 - C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\xgb_feature_names.json
 - C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\xgb_model_config.json
 - C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\xgb_summary.json
All required Notebook 05 artifacts for explainability are present.


In [ ]:
# SHAP — Hybrid model (XGBoost) explanations by horizon
import json
import numpy as np
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from IPython.display import Image, display

SHAP_DIR = PROJECT_ROOT / "artifacts" / "explainability" / "shap"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

def load_feature_names(path: Path) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    # Support both formats: list[str] or {"features": [...]}
    if isinstance(payload, list):
        names = payload
    elif isinstance(payload, dict) and "features" in payload and isinstance(payload["features"], list):
        names = payload["features"]
    else:
        raise ValueError(f"Unrecognised feature-name format in {path}")
    return [str(n) for n in names]

X = np.load(HYBRID_DIR / "stack_test_X.npy", allow_pickle=True)
X = np.asarray(X, dtype=float)

if X.ndim != 2:
    raise ValueError(f"Expected 2D design matrix stack_test_X.npy; got shape {X.shape}")

# Prefer X-matching feature names; fall back between available files
candidates = [
    HYBRID_DIR / "feature_names.json",          # often matches stack_test_X columns
    HYBRID_DIR / "xgb_feature_names.json",      # sometimes stored as metadata only
    HYBRID_DIR / "feature_names.json",
]
feature_names = None
for fp in candidates:
    if fp.exists():
        names = load_feature_names(fp)
        if len(names) == X.shape[1]:
            feature_names = names
            feature_source = fp
            break

if feature_names is None:
    # Load for diagnostics and raise a clear error
    available = []
    for fp in [HYBRID_DIR / "feature_names.json", HYBRID_DIR / "xgb_feature_names.json"]:
        if fp.exists():
            available.append((str(fp), len(load_feature_names(fp))))
    raise ValueError(
        "Could not find feature-name list matching stack_test_X columns. "
        f"stack_test_X has {X.shape[1]} columns; available feature lists: {available}"
    )

print("Using feature names from:", feature_source)

# Use a fixed-size sample for speed and reproducibility
rng = np.random.default_rng(42)
n = X.shape[0]
sample_size = int(min(500, n))
idx = rng.choice(n, size=sample_size, replace=False) if n > sample_size else np.arange(n)
X_sample = X[idx]

X_df = pd.DataFrame(X_sample, columns=feature_names)

def load_booster(model_path: Path) -> xgb.Booster:
    booster = xgb.Booster()
    booster.load_model(str(model_path))
    return booster

def shap_for_horizon(h: int, model_path: Path, show_inline: bool = True) -> None:
    booster = load_booster(model_path)
    explainer = shap.TreeExplainer(booster)
    shap_values = explainer.shap_values(X_df)

    # Global importance table
    mean_abs = np.mean(np.abs(shap_values), axis=0)
    imp = (
        pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )
    imp_path = SHAP_DIR / f"shap_importance_h{h}.csv"
    imp.to_csv(imp_path, index=False)

    # Summary (beeswarm)
    plt.figure()
    shap.summary_plot(shap_values, X_df, show=False)
    beeswarm_path = SHAP_DIR / f"shap_beeswarm_h{h}.png"
    plt.gcf().savefig(beeswarm_path, dpi=200, bbox_inches="tight")
    plt.close()

    # Summary (bar)
    plt.figure()
    shap.summary_plot(shap_values, X_df, plot_type="bar", show=False)
    bar_path = SHAP_DIR / f"shap_bar_h{h}.png"
    plt.gcf().savefig(bar_path, dpi=200, bbox_inches="tight")
    plt.close()

    print(f"Horizon t+{h}: saved {imp_path.name}, {beeswarm_path.name}, {bar_path.name}")
    if show_inline:
        display(Image(filename=str(bar_path)))
        display(Image(filename=str(beeswarm_path)))

models = {
    1: HYBRID_DIR / "xgb_t1.json",
    3: HYBRID_DIR / "xgb_t3.json",
    5: HYBRID_DIR / "xgb_t5.json",
}

for h, p in models.items():
    shap_for_horizon(h, p, show_inline=True)

Using feature names from: C:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\results\hybrid\feature_names.json
Horizon t+1: saved shap_importance_h1.csv, shap_beeswarm_h1.png, shap_bar_h1.png
Horizon t+3: saved shap_importance_h3.csv, shap_beeswarm_h3.png, shap_bar_h3.png
Horizon t+5: saved shap_importance_h5.csv, shap_beeswarm_h5.png, shap_bar_h5.png


In [ ]:
# LIME — Hybrid model (XGBoost) local explanations by horizon
from IPython.display import HTML, display
from lime.lime_tabular import LimeTabularExplainer

LIME_DIR = PROJECT_ROOT / "artifacts" / "explainability" / "lime"
LIME_DIR.mkdir(parents=True, exist_ok=True)

# Use the same sampled matrix as background for perturbations
explainer = LimeTabularExplainer(
    training_data=X_sample,
    feature_names=feature_names,
    mode="regression",
    discretize_continuous=True,
    random_state=42,
)

def predict_with_booster(booster: xgb.Booster, x_numpy: np.ndarray) -> np.ndarray:
    x_numpy = np.asarray(x_numpy, dtype=float)
    dmat = xgb.DMatrix(x_numpy, feature_names=feature_names)
    yhat = booster.predict(dmat)
    return np.asarray(yhat, dtype=float)

def lime_for_horizon(h: int, model_path: Path, instance_indices: list[int], show_inline: bool = True) -> None:
    booster = load_booster(model_path)
    predict_fn = lambda x: predict_with_booster(booster, x)

    for j, i in enumerate(instance_indices):
        x_i = X_sample[i]
        exp = explainer.explain_instance(x_i, predict_fn, num_features=12)
        html = exp.as_html()
        out_path = LIME_DIR / f"lime_h{h}_instance_{i}.html"
        out_path.write_text(html, encoding="utf-8")
        print(f"Horizon t+{h}: saved {out_path.name}")

        # Show only the first explanation per horizon inline to keep output readable
        if show_inline and j == 0:
            display(HTML(f"<h3>LIME explanation — Horizon t+{h}, instance {i}</h3>"))
            display(HTML(html))

# Explain a small number of representative instances for each horizon
instance_indices = list(range(min(3, X_sample.shape[0])))
for h, p in models.items():
    lime_for_horizon(h, p, instance_indices, show_inline=True)

Horizon t+1: saved lime_h1_instance_0.html
Horizon t+1: saved lime_h1_instance_1.html
Horizon t+1: saved lime_h1_instance_2.html
Horizon t+3: saved lime_h3_instance_0.html
Horizon t+3: saved lime_h3_instance_1.html
Horizon t+3: saved lime_h3_instance_2.html
Horizon t+5: saved lime_h5_instance_0.html
Horizon t+5: saved lime_h5_instance_1.html
Horizon t+5: saved lime_h5_instance_2.html
